# Actividad 2.1 — Valores Nulos
**Alumno:** Gabriel · A01736195
**Dataset:** Citas_Digital_Filtrado.csv

**Objetivo:** identificar los valores nulos por columna y aplicar los métodos de sustitución vistos en clase (media, mediana, número concreto, string concreto, forward fill y backward fill). Al final comparamos contra el método de eliminación (dropna) y exportamos el CSV final.

In [17]:
import pandas as pd
import numpy as np

data = pd.read_csv('Citas_Digital_Filtrado.csv')
print(f"Filas: {data.shape[0]} | Columnas: {data.shape[1]}")

# ¿Cuántos valores nulos tiene cada columna?
data.isnull().sum()

Filas: 676 | Columnas: 16


BDC                                          16
Fecha                                       601
Nombre Cliente                               21
Telefono                                    674
Estatus de Lead                              35
Potencial de compra                          74
PDM                                           1
SDC                                           2
Venta                                         5
Asesor Asignado                              32
¿Por que no ha visitado la agencia?         183
¿Por que no hay solicitud de credito?       297
¿Por que no hubo prueba de manejo ?         321
¿Por que no hubo proceso wow?               312
¿Por que se asigno antes de vistar piso?    323
Unnamed: 15                                 660
dtype: int64

In [18]:
# Algunos nombres de columna traen espacios invisibles al final (p. ej. 'BDC ')
# Si no se quitan, pandas no encuentra la columna cuando la llamamos por su nombre
data.columns = data.columns.astype(str).str.strip()
data.columns.tolist()

['BDC',
 'Fecha',
 'Nombre Cliente',
 'Telefono',
 'Estatus de Lead',
 'Potencial de compra',
 'PDM',
 'SDC',
 'Venta',
 'Asesor Asignado',
 '¿Por que no ha visitado la agencia?',
 '¿Por que no hay solicitud de credito?',
 '¿Por que no hubo prueba de manejo ?',
 '¿Por que no hubo proceso wow?',
 '¿Por que se asigno antes de vistar piso?',
 'Unnamed: 15']

In [19]:
# CUARTO MÉTODO: string concreto
# Columnas de texto donde el vacío significa "el dato no se capturó"
data['BDC'] = data['BDC'].fillna('Sin registro')
data['Nombre Cliente'] = data['Nombre Cliente'].fillna('Sin registro')
data['Telefono'] = data['Telefono'].fillna('Sin registro')
data['Asesor Asignado'] = data['Asesor Asignado'].fillna('Sin asignar')
data['Fecha'] = data['Fecha'].fillna('Sin fecha')
data['Unnamed: 15'] = data['Unnamed: 15'].fillna('Sin comentarios')

# Las 5 columnas de "¿Por que...?" solo se llenan cuando NO se realizó la actividad;
# si están vacías es porque SÍ se realizó, así que se etiqueta como "No aplica"
for col in [c for c in data.columns if c.startswith('¿Por que')]:
    data[col] = data[col].fillna('No aplica (sí se realizó)')

# Corroboramos valores nulos
data.isnull().sum()

BDC                                          0
Fecha                                        0
Nombre Cliente                               0
Telefono                                     0
Estatus de Lead                             35
Potencial de compra                         74
PDM                                          1
SDC                                          2
Venta                                        5
Asesor Asignado                              0
¿Por que no ha visitado la agencia?          0
¿Por que no hay solicitud de credito?        0
¿Por que no hubo prueba de manejo ?          0
¿Por que no hubo proceso wow?                0
¿Por que se asigno antes de vistar piso?     0
Unnamed: 15                                  0
dtype: int64

In [20]:
# QUINTO MÉTODO: forward fill (rellena con el último valor válido hacia abajo)
# Estatus de Lead: el estatus de un lead suele repetirse del registro anterior
data['Estatus de Lead'] = data['Estatus de Lead'].ffill()

# Corroboramos valores nulos
data.isnull().sum()

BDC                                          0
Fecha                                        0
Nombre Cliente                               0
Telefono                                     0
Estatus de Lead                              0
Potencial de compra                         74
PDM                                          1
SDC                                          2
Venta                                        5
Asesor Asignado                              0
¿Por que no ha visitado la agencia?          0
¿Por que no hay solicitud de credito?        0
¿Por que no hubo prueba de manejo ?          0
¿Por que no hubo proceso wow?                0
¿Por que se asigno antes de vistar piso?     0
Unnamed: 15                                  0
dtype: int64

In [21]:
# SEGUNDO MÉTODO: mediana
# 'Potencial de compra' es numérica, pero viene sucia: trae textos como 'Ventas ', '80%%', "8'%", '80&'
# Primero la limpiamos y la convertimos a número:
data['Potencial de compra'] = data['Potencial de compra'].astype(str).str.strip()
for basura in ['Ventas', '%', "'", '&']:
    data['Potencial de compra'] = data['Potencial de compra'].str.replace(basura, '', regex=False)
data['Potencial de compra'] = data['Potencial de compra'].str.strip()
data['Potencial de compra'] = data['Potencial de compra'].replace(['nan', ''], np.nan)
data['Potencial de compra'] = data['Potencial de compra'].astype(float)

# Elegimos la MEDIANA porque es robusta a valores extremos:
# unos cuantos leads con potencial altísimo inflarían el promedio (media)
mediana = round(data['Potencial de compra'].median(), 2)
print(f"Mediana de Potencial de compra: {mediana}")
data['Potencial de compra'] = data['Potencial de compra'].fillna(mediana)

# Corroboramos valores nulos
data.isnull().sum()

Mediana de Potencial de compra: 0.75


BDC                                         0
Fecha                                       0
Nombre Cliente                              0
Telefono                                    0
Estatus de Lead                             0
Potencial de compra                         0
PDM                                         1
SDC                                         2
Venta                                       5
Asesor Asignado                             0
¿Por que no ha visitado la agencia?         0
¿Por que no hay solicitud de credito?       0
¿Por que no hubo prueba de manejo ?         0
¿Por que no hubo proceso wow?               0
¿Por que se asigno antes de vistar piso?    0
Unnamed: 15                                 0
dtype: int64

In [22]:
# TERCER MÉTODO: número concreto (0)
# PDM, SDC y Venta son banderas de proceso: si están vacías significa que NO se realizó, o sea 0
data['PDM'] = data['PDM'].fillna(0)
data['SDC'] = data['SDC'].fillna(0)
data['Venta'] = data['Venta'].fillna(0)

# Corroboramos valores nulos
data.isnull().sum()

BDC                                         0
Fecha                                       0
Nombre Cliente                              0
Telefono                                    0
Estatus de Lead                             0
Potencial de compra                         0
PDM                                         0
SDC                                         0
Venta                                       0
Asesor Asignado                             0
¿Por que no ha visitado la agencia?         0
¿Por que no hay solicitud de credito?       0
¿Por que no hubo prueba de manejo ?         0
¿Por que no hubo proceso wow?               0
¿Por que se asigno antes de vistar piso?    0
Unnamed: 15                                 0
dtype: int64

In [23]:
# MÉTODO DE ELIMINACIÓN (solo como comparación, NO lo aplicamos)
data_original = pd.read_csv('Citas_Digital_Filtrado.csv')
print(f"Filas originales: {len(data_original)}")
print(f"Filas que quedarían con dropna(): {len(data_original.dropna())}")
print(f"Filas que conservamos con nuestros métodos: {len(data)}")
# Conclusión: con dropna() el dataset quedaría en 0 filas (todas tienen al menos un nulo),
# por eso aquí la estrategia correcta es SUSTITUIR y no eliminar.

Filas originales: 676
Filas que quedarían con dropna(): 0
Filas que conservamos con nuestros métodos: 676


In [24]:
# Exportamos el dataset final, ya sin valores nulos
data.to_csv('Citas_Digital_Filtrado_sin_nulos.csv', index=False)
print("Archivo exportado: Citas_Digital_Filtrado_sin_nulos.csv")
print(f"Filas: {len(data)} | Nulos totales: {data.isnull().sum().sum()}")

Archivo exportado: Citas_Digital_Filtrado_sin_nulos.csv
Filas: 676 | Nulos totales: 0


## Descripción de técnicas por columna

| Columna | Nulos | Método | Reemplazo | Justificación |
|---|---|---|---|---|
| BDC | 16 | 4. String concreto | 'Sin registro' | Dato de texto no capturado. |
| Fecha | 601 | 4. String concreto | 'Sin fecha' | 89% vacía; una fecha no se promedia ni se inventa, se etiqueta. |
| Nombre Cliente | 21 | 4. String concreto | 'Sin registro' | Dato de texto no capturado. |
| Telefono | 674 | 4. String concreto | 'Sin registro' | 99.7% vacía; se conserva la columna con etiqueta. |
| Estatus de Lead | 35 | 5. Forward fill | último valor válido | El estatus tiende a repetirse entre registros consecutivos. |
| Potencial de compra | 74 | 2. Mediana | 0.75 | Numérica con valores extremos; la mediana es robusta. Antes se limpió basura ('Ventas', '%', ''', '&'). |
| PDM | 1 | 3. Número concreto | 0 | Bandera de proceso: vacío = no se realizó. |
| SDC | 2 | 3. Número concreto | 0 | Bandera de proceso: vacío = no se realizó. |
| Venta | 5 | 3. Número concreto | 0 | Bandera de proceso: vacío = no hubo venta. |
| Asesor Asignado | 32 | 4. String concreto | 'Sin asignar' | El vacío significa que aún no se asigna asesor. |
| ¿Por que no ha visitado la agencia? | 183 | 4. String concreto | 'No aplica (sí se realizó)' | La pregunta solo aplica cuando NO se hizo la actividad. |
| ¿Por que no hay solicitud de credito? | 297 | 4. String concreto | 'No aplica (sí se realizó)' | Misma lógica. |
| ¿Por que no hubo prueba de manejo? | 321 | 4. String concreto | 'No aplica (sí se realizó)' | Misma lógica. |
| ¿Por que no hubo proceso wow? | 312 | 4. String concreto | 'No aplica (sí se realizó)' | Misma lógica. |
| ¿Por que se asigno antes de vistar piso? | 323 | 4. String concreto | 'No aplica (sí se realizó)' | Misma lógica. |
| Unnamed: 15 | 660 | 4. String concreto | 'Sin comentarios' | Columna de comentarios sin encabezado. |

**Resultado:** 676 filas conservadas, 0 valores nulos. Con dropna() el dataset habría quedado en 0 filas.